# OSACT5 Dataset: Download, Inspection, Cleaning & Validation

**A reproducible preprocessing pipeline for Arabic offensive-language / hate-speech research (EMNLP-quality)**

This notebook downloads the OSACT2022 (OSACT5) shared-task dataset, performs a full
data-quality audit, implements a configurable Arabic text-normalization pipeline, and
exports clean, analysis-ready CSV files together with a research-grade summary report.


## Part 1 — Environment Setup

In [ ]:
!pip -q install emoji==2.12.1 ftfy==6.2.0 chardet==5.2.0 tqdm==4.66.4

In [ ]:
import os
import re
import sys
import csv
import json
import random
import hashlib
import unicodedata
import platform
import subprocess
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import emoji
import ftfy
import chardet
from tqdm.auto import tqdm

tqdm.pandas()


In [ ]:
SEED = 42

def set_global_seed(seed: int = SEED) -> None:
    '''Fix all sources of randomness we touch in this notebook.'''
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_global_seed(SEED)
print(f"Global seed fixed to {SEED}")


In [ ]:
# Environment / version report
def print_environment_report() -> None:
    '''Print a reproducibility-grade environment report (paper appendix material).'''
    print("=" * 60)
    print("ENVIRONMENT REPORT")
    print("=" * 60)
    print(f"Python        : {platform.python_version()}")
    print(f"Platform      : {platform.platform()}")
    print(f"pandas        : {pd.__version__}")
    print(f"numpy         : {np.__version__}")
    print(f"matplotlib    : {plt.matplotlib.__version__}")
    print(f"emoji         : {emoji.__version__}")
    print(f"ftfy          : {ftfy.__version__}")
    print(f"chardet       : {chardet.__version__}")
    print("=" * 60)

print_environment_report()


## Part 2 — Download the OSACT5 Dataset

Download the three official OSACT2022 shared-task files with `wget`, save them under
standardized local filenames, and verify that every download actually succeeded
(non-zero size, correct HTTP status, file present on disk).


In [ ]:
# Dataset locations
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

DATASET_URLS = {
    "train": "https://alt.qcri.org/resources/OSACT2022/OSACT2022-sharedTask-train.txt",
    "dev":   "https://alt.qcri.org/resources/OSACT2022/OSACT2022-sharedTask-dev.txt",
    "test":  "https://alt.qcri.org/resources/OSACT2022/OSACT2022-sharedTask-test-tweets.txt",
}

LOCAL_PATHS = {
    "train": os.path.join(DATA_DIR, "train.txt"),
    "dev":   os.path.join(DATA_DIR, "dev.txt"),
    "test":  os.path.join(DATA_DIR, "test.txt"),
}


In [ ]:
def download_file(url: str, out_path: str, force: bool = False) -> bool:
    '''Download `url` to `out_path` with wget and return True on verified success.'''
    if os.path.exists(out_path) and not force:
        print(f"[skip] {out_path} already exists ({os.path.getsize(out_path)} bytes)")
        return True

    cmd = ["wget", "-q", "--show-progress", "-O", out_path, url]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"[FAIL] wget exited with code {result.returncode} for {url}")
        print(result.stderr[-500:])
        return False

    return verify_download(out_path)


def verify_download(path: str) -> bool:
    '''A download is considered successful iff the file exists and is non-empty.'''
    if not os.path.exists(path):
        print(f"[FAIL] {path} was not created")
        return False
    size = os.path.getsize(path)
    if size == 0:
        print(f"[FAIL] {path} is empty (0 bytes)")
        return False
    print(f"[OK]   {path} downloaded successfully ({size:,} bytes)")
    return True


download_status = {}
for split, url in DATASET_URLS.items():
    print(f"\nDownloading '{split}' split...")
    download_status[split] = download_file(url, LOCAL_PATHS[split])

print("\n" + "=" * 60)
print("DOWNLOAD SUMMARY")
print("=" * 60)
for split, ok in download_status.items():
    print(f"{split:6s}: {'SUCCESS' if ok else 'FAILED'}")

if not all(download_status.values()):
    raise RuntimeError(
        "One or more downloads failed. If you are running outside a network with access "
        "to alt.qcri.org, manually place train.txt / dev.txt / test.txt inside the "
        f"'{DATA_DIR}/' directory and re-run this cell (force=False will detect them)."
    )


## Part 3 — Load the Data

OSACT-style release files are typically tab-separated, but we do not hard-code this
assumption: we auto-detect the delimiter from the raw file using `csv.Sniffer`, with a
robust fallback heuristic (tab vs. comma frequency) if sniffing fails on messy tweet text.


In [ ]:
def detect_delimiter(path: str, sample_lines: int = 20) -> str:
    '''Auto-detect the field delimiter of a raw text file.

    Strategy:
      1. Try csv.Sniffer on a sample of lines.
      2. Fall back to comparing tab vs comma counts per line (tweets often contain
         literal commas in text, so tab is preferred when both are plausible).

    Robust to files shorter than `sample_lines` (uses itertools.islice instead of a
    fixed-size list comprehension with next(), which used to raise StopIteration on
    small files such as dev/test splits).
    '''
    from itertools import islice
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        sample = "".join(islice(f, sample_lines))

    if not sample.strip():
        raise ValueError(f"{path} appears to be empty — cannot detect delimiter")

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters="\t,;|")
        return dialect.delimiter
    except csv.Error:
        pass

    tab_count = sample.count("\t")
    comma_count = sample.count(",")
    if tab_count >= comma_count:
        return "\t"
    return ","


def load_split(path: str) -> pd.DataFrame:
    '''Load a raw OSACT split with auto-detected delimiter and robust parsing.'''
    delimiter = detect_delimiter(path)
    print(f"[{os.path.basename(path)}] detected delimiter: {repr(delimiter)}")

    df = pd.read_csv(
        path,
        sep=delimiter,
        engine="python",
        quoting=csv.QUOTE_NONE,
        encoding="utf-8",
        on_bad_lines="warn",
        dtype=str,          # keep everything as string first; we cast later where needed
        keep_default_na=False,
    )
    # Normalize column names (strip whitespace, lowercase-safe copy kept separately)
    df.columns = [c.strip() for c in df.columns]
    return df


raw = {}
for split, path in LOCAL_PATHS.items():
    raw[split] = load_split(path)

print("\nLoaded splits:", {k: v.shape for k, v in raw.items()})


In [ ]:
def describe_split(name: str, df: pd.DataFrame) -> None:
    print("=" * 60)
    print(f"SPLIT: {name}")
    print("=" * 60)
    print(f"Shape       : {df.shape}")
    print(f"Columns     : {list(df.columns)}")
    print("\nDtypes:")
    print(df.dtypes)
    print("\nFirst rows:")
    display(df.head())
    print()

for split, df in raw.items():
    describe_split(split, df)


## Part 4 — Data Inspection

We generate a full data-quality report per split: missing values, duplicate rows/texts/IDs,
empty texts, text-length statistics, Arabic vs. Latin character ratios, and counts of
emojis, URLs, hashtags, and mentions. We then plot label distributions and text-length
histograms.

We try to auto-detect the text column and (if present) an ID column and label column(s),
since exact OSACT column naming can vary slightly between releases.


In [ ]:
# Column auto-detection helpers
ID_CANDIDATES = ["tweet_id", "id", "twitter_id"]
TEXT_CANDIDATES = ["text", "tweet", "tweet_text", "content"]
LABEL_CANDIDATES = ["off_label", "hs_label", "offensive_label", "label", "off", "hs",
                     "hate_speech_label", "class"]

def find_column(columns, candidates):
    lower_map = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    return None

def resolve_schema(df: pd.DataFrame) -> dict:
    id_col = find_column(df.columns, ID_CANDIDATES)
    text_col = find_column(df.columns, TEXT_CANDIDATES)
    label_cols = [c for c in df.columns if c.lower() in LABEL_CANDIDATES]

    # Fallbacks: if nothing matched, assume col0 = id, col1 = text (common OSACT layout)
    if text_col is None:
        text_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]
    if id_col is None and len(df.columns) > 1:
        id_col = df.columns[0]

    return {"id_col": id_col, "text_col": text_col, "label_cols": label_cols}

schemas = {split: resolve_schema(df) for split, df in raw.items()}
for split, schema in schemas.items():
    print(f"{split:6s} -> {schema}")


In [ ]:
# Regex patterns for surface-feature counting
URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\S+")
ARABIC_CHAR_RE = re.compile(r"[\u0600-\u06FF\u0750-\u077F]")
LATIN_CHAR_RE = re.compile(r"[A-Za-z]")

def count_emojis(text: str) -> int:
    return len([ch for ch in text if ch in emoji.EMOJI_DATA])

def inspect_split(name: str, df: pd.DataFrame, schema: dict) -> dict:
    text_col = schema["text_col"]
    id_col = schema["id_col"]
    texts = df[text_col].astype(str)

    lengths_chars = texts.str.len()
    lengths_words = texts.str.split().apply(len)

    report = {
        "split": name,
        "n_rows": len(df),
        "missing_values_total": int(df.isna().sum().sum()) + int((df == "").sum().sum()),
        "missing_values_by_column": {c: int((df[c].isna() | (df[c] == "")).sum()) for c in df.columns},
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_texts": int(texts.duplicated().sum()),
        "duplicate_ids": int(df[id_col].duplicated().sum()) if id_col else None,
        "empty_texts": int((texts.str.strip() == "").sum()),
        "text_length_chars": {
            "mean": float(lengths_chars.mean()),
            "median": float(lengths_chars.median()),
            "std": float(lengths_chars.std()),
            "min": int(lengths_chars.min()),
            "max": int(lengths_chars.max()),
        },
        "text_length_words": {
            "mean": float(lengths_words.mean()),
            "median": float(lengths_words.median()),
            "std": float(lengths_words.std()),
        },
        "arabic_char_ratio": float(texts.apply(lambda t: len(ARABIC_CHAR_RE.findall(t)) / max(len(t), 1)).mean()),
        "latin_char_ratio": float(texts.apply(lambda t: len(LATIN_CHAR_RE.findall(t)) / max(len(t), 1)).mean()),
        "total_emojis": int(texts.apply(count_emojis).sum()),
        "total_urls": int(texts.apply(lambda t: len(URL_RE.findall(t))).sum()),
        "total_hashtags": int(texts.apply(lambda t: len(HASHTAG_RE.findall(t))).sum()),
        "total_mentions": int(texts.apply(lambda t: len(MENTION_RE.findall(t))).sum()),
    }
    return report

inspection_reports = {}
for split, df in raw.items():
    inspection_reports[split] = inspect_split(split, df, schemas[split])
    print("=" * 60)
    print(f"INSPECTION REPORT: {split}")
    print("=" * 60)
    print(json.dumps(inspection_reports[split], indent=2, ensure_ascii=False))
    print()


In [ ]:
# Plot: label distributions
def plot_label_distributions(df: pd.DataFrame, schema: dict, split_name: str) -> None:
    label_cols = schema["label_cols"]
    if not label_cols:
        print(f"[{split_name}] no label columns detected — skipping label plot "
              f"(expected for the unlabeled test split).")
        return

    n = len(label_cols)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, col in zip(axes, label_cols):
        counts = df[col].value_counts()
        ax.bar(counts.index.astype(str), counts.values, color="#3B6E8F")
        ax.set_title(f"{split_name}: {col}")
        ax.set_xlabel("Class")
        ax.set_ylabel("Count")
        for tick in ax.get_xticklabels():
            tick.set_rotation(30)

    plt.tight_layout()
    plt.show()

for split, df in raw.items():
    plot_label_distributions(df, schemas[split], split)


In [ ]:
# Plot: text length histograms
def plot_length_histogram(df: pd.DataFrame, schema: dict, split_name: str) -> None:
    text_col = schema["text_col"]
    lengths = df[text_col].astype(str).str.len()

    plt.figure(figsize=(7, 4))
    plt.hist(lengths, bins=40, color="#8F3B5C", edgecolor="white")
    plt.title(f"{split_name}: text length distribution (characters)")
    plt.xlabel("Characters per tweet")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

for split, df in raw.items():
    plot_length_histogram(df, schemas[split], split)


## Part 5 — Dataset Validation

We now formally validate each split and print an itemized anomaly report. Checks:

- No structurally corrupted rows (wrong field count vs. header)
- All text is valid UTF-8
- ID column is unique (where present)
- Label values are within the expected/consistent set
- Class distribution is reported for sanity-checking imbalance


In [ ]:
def is_valid_utf8(s: str) -> bool:
    try:
        s.encode("utf-8").decode("utf-8")
        return True
    except (UnicodeEncodeError, UnicodeDecodeError):
        return False


def validate_split(name: str, path: str, df: pd.DataFrame, schema: dict) -> dict:
    anomalies = []

    # 1. Corrupted rows: re-scan raw file for lines whose field count != header field count
    delimiter = detect_delimiter(path)
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()
    header_fields = len(lines[0].split(delimiter))
    corrupted_lines = [
        i for i, line in enumerate(lines[1:], start=2)
        if len(line.rstrip("\n").split(delimiter)) != header_fields
    ]
    if corrupted_lines:
        anomalies.append(f"{len(corrupted_lines)} structurally corrupted line(s), "
                          f"e.g. lines {corrupted_lines[:5]}")

    # 2. UTF-8 validity
    text_col = schema["text_col"]
    invalid_utf8 = (~df[text_col].astype(str).apply(is_valid_utf8)).sum()
    if invalid_utf8:
        anomalies.append(f"{invalid_utf8} row(s) with invalid UTF-8 text")

    # 3. Unique IDs
    id_col = schema["id_col"]
    if id_col:
        dup_ids = df[id_col].duplicated().sum()
        if dup_ids:
            anomalies.append(f"{dup_ids} duplicated ID(s) in column '{id_col}'")
    else:
        anomalies.append("no ID column detected — uniqueness check skipped")

    # 4. Label consistency
    label_report = {}
    for col in schema["label_cols"]:
        values = df[col].value_counts(dropna=False)
        label_report[col] = values.to_dict()
        # Flag stray/rare label values (possible typos), heuristic: any value with <0.1% share
        rare = values[values < max(1, int(0.001 * len(df)))]
        if len(rare) > 0 and len(values) > 2:
            anomalies.append(f"column '{col}' has suspicious rare label values: {rare.to_dict()}")

    result = {
        "split": name,
        "n_corrupted_rows": len(corrupted_lines),
        "n_invalid_utf8": int(invalid_utf8),
        "n_duplicate_ids": int(df[id_col].duplicated().sum()) if id_col else None,
        "label_distribution": label_report,
        "anomalies": anomalies if anomalies else ["none detected"],
    }
    return result


validation_reports = {}
for split, df in raw.items():
    validation_reports[split] = validate_split(split, LOCAL_PATHS[split], df, schemas[split])
    print("=" * 60)
    print(f"VALIDATION REPORT: {split}")
    print("=" * 60)
    print(json.dumps(validation_reports[split], indent=2, ensure_ascii=False))
    print()


## Part 6 — Arabic Preprocessing Pipeline

A fully **configurable**, function-based pipeline. Every transformation is independently
toggleable via `PreprocessConfig`, so the exact preprocessing recipe used in the paper can
be stated unambiguously (and ablated).

**Design choices worth documenting in the paper:**

- We normalize Unicode (NFKC) to collapse compatibility variants (e.g., presentation-form
  Arabic letters) into canonical codepoints — this reduces spurious vocabulary sparsity
  without altering meaning.
- We normalize Alef variants (أ إ آ ٱ → ا) and Yeh variants (ى ئ → ي) because these are
  extremely common *orthographic* (not semantic) variations in social-media Arabic, and
  normalizing them is standard practice in Arabic NLP (e.g., Farasa, CAMeL Tools) — it
  reduces sparsity without changing meaning.
- **Teh Marbuta normalization (ة → ه) is OFF by default** and only justified if a
  downstream task is robust to gender/word-final ambiguity (ة often marks feminine nouns
  and merging it with ه can lose morphological signal). We expose it as an explicit flag
  and document the trade-off rather than silently applying it.
- We deliberately **do not** remove hashtags, emojis, punctuation, or numbers, since these
  frequently carry sentiment/offensive-language signal (e.g., emoji-based insults,
  hashtag topics, exclamation marks as intensifiers) that is directly relevant to hate
  speech / offensive language detection.


In [ ]:
from dataclasses import dataclass

@dataclass
class PreprocessConfig:
    '''Toggle each Arabic preprocessing step independently.'''
    unicode_normalize: bool = True        # NFKC normalization
    fix_encoding: bool = True             # ftfy mojibake repair
    remove_tatweel: bool = True           # remove Arabic kashida (ـ)
    remove_diacritics: bool = True        # remove tashkeel (َ ً ُ ٌ ِ ٍ ّ ْ)
    normalize_alef: bool = True           # أ إ آ ٱ -> ا
    normalize_yeh: bool = True            # ى ئ -> ي
    normalize_teh_marbuta: bool = False   # ة -> ه  (OFF by default, see markdown above)
    remove_urls: bool = True
    remove_mentions: bool = True
    remove_html: bool = True
    remove_extra_whitespace: bool = True

    # explicitly preserved (documented for clarity, not actually toggled off):
    # hashtags, emojis, punctuation, numbers


In [ ]:
# Individual, composable preprocessing functions

TATWEEL_RE = re.compile(r"\u0640")
DIACRITICS_RE = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED]")
ALEF_VARIANTS_RE = re.compile(r"[\u0622\u0623\u0625\u0671]")   # آ أ إ ٱ
YEH_VARIANTS_RE = re.compile(r"[\u0649\u0626]")                # ى ئ
TEH_MARBUTA_RE = re.compile(r"\u0629")                          # ة
HTML_TAG_RE = re.compile(r"<[^>]+>")
HTML_ENTITY_RE = re.compile(r"&[a-zA-Z#0-9]+;")
WHITESPACE_RE = re.compile(r"\s+")


def normalize_unicode(text: str) -> str:
    '''NFKC-normalize to collapse Unicode compatibility variants.'''
    return unicodedata.normalize("NFKC", text)


def fix_encoding(text: str) -> str:
    '''Repair mojibake / encoding artifacts (e.g. double-encoded UTF-8).'''
    return ftfy.fix_text(text)


def remove_tatweel(text: str) -> str:
    '''Remove the Arabic elongation character (kashida), purely stylistic.'''
    return TATWEEL_RE.sub("", text)


def remove_diacritics(text: str) -> str:
    '''Remove tashkeel (short vowel marks) — rarely used in social media text anyway.'''
    return DIACRITICS_RE.sub("", text)


def normalize_alef(text: str) -> str:
    '''Collapse Alef variants (أ إ آ ٱ) to bare Alef (ا). Orthographic, not semantic.'''
    return ALEF_VARIANTS_RE.sub("\u0627", text)


def normalize_yeh(text: str) -> str:
    '''Collapse Yeh variants (ى ئ) to standard Yeh (ي). Orthographic, not semantic.'''
    return YEH_VARIANTS_RE.sub("\u064A", text)


def normalize_teh_marbuta(text: str) -> str:
    '''Collapse Teh Marbuta (ة) to Heh (ه).

    CAUTION: this can remove a morphological (often gender) signal. Only enable this if
    your downstream task/tokenizer is known to benefit (e.g. matches a stemmed lexicon).
    '''
    return TEH_MARBUTA_RE.sub("\u0647", text)


def remove_urls(text: str) -> str:
    return URL_RE.sub(" ", text)


def remove_mentions(text: str) -> str:
    return MENTION_RE.sub(" ", text)


def remove_html(text: str) -> str:
    text = HTML_TAG_RE.sub(" ", text)
    text = HTML_ENTITY_RE.sub(" ", text)
    return text


def remove_extra_whitespace(text: str) -> str:
    return WHITESPACE_RE.sub(" ", text).strip()


In [ ]:
# Master pipeline: applies enabled steps in a fixed, documented order

def clean_text(text: str, config: PreprocessConfig) -> str:
    '''Apply the configured Arabic preprocessing pipeline to a single string.

    Order matters: encoding/unicode fixes first, structural removals (URLs/mentions/HTML)
    next, then character-level normalization, then final whitespace cleanup.
    Hashtags, emojis, punctuation, and numbers are never touched.
    '''
    if not isinstance(text, str):
        return text

    if config.fix_encoding:
        text = fix_encoding(text)
    if config.unicode_normalize:
        text = normalize_unicode(text)
    if config.remove_html:
        text = remove_html(text)
    if config.remove_urls:
        text = remove_urls(text)
    if config.remove_mentions:
        text = remove_mentions(text)
    if config.remove_tatweel:
        text = remove_tatweel(text)
    if config.remove_diacritics:
        text = remove_diacritics(text)
    if config.normalize_alef:
        text = normalize_alef(text)
    if config.normalize_yeh:
        text = normalize_yeh(text)
    if config.normalize_teh_marbuta:
        text = normalize_teh_marbuta(text)
    if config.remove_extra_whitespace:
        text = remove_extra_whitespace(text)

    return text


def clean_dataframe(df: pd.DataFrame, text_col: str, config: PreprocessConfig) -> pd.DataFrame:
    '''Return a copy of df with an added 'text_clean' column.'''
    out = df.copy()
    out["text_clean"] = out[text_col].astype(str).progress_apply(lambda t: clean_text(t, config))
    return out


# Instantiate the default configuration used for this paper's experiments
default_config = PreprocessConfig()
print(default_config)


## Part 7 — Cleaning Report

We apply the pipeline to every split, show concrete before/after examples, count how many
rows were actually altered by cleaning, and summarize the effect of each individual step.


In [ ]:
cleaned = {}
for split, df in raw.items():
    text_col = schemas[split]["text_col"]
    cleaned[split] = clean_dataframe(df, text_col, default_config)
    n_changed = (cleaned[split]["text_clean"] != cleaned[split][text_col]).sum()
    print(f"[{split}] rows changed by cleaning: {n_changed} / {len(cleaned[split])} "
          f"({n_changed / len(cleaned[split]):.1%})")


In [ ]:
# Before / after qualitative examples
def show_before_after(df: pd.DataFrame, text_col: str, n: int = 5, seed: int = SEED) -> None:
    changed = df[df["text_clean"] != df[text_col]]
    sample = changed.sample(min(n, len(changed)), random_state=seed) if len(changed) else changed
    for i, row in sample.iterrows():
        print("-" * 60)
        print("BEFORE:", row[text_col])
        print("AFTER :", row["text_clean"])

print("Example transformations (train split):")
show_before_after(cleaned["train"], schemas["train"]["text_col"])


In [ ]:
# Per-step impact summary table
def summarize_step_impact(df: pd.DataFrame, text_col: str, config: PreprocessConfig) -> pd.DataFrame:
    '''For each enabled step, count how many rows it individually alters (single-step ablation).'''
    base_texts = df[text_col].astype(str)
    steps = {
        "fix_encoding": fix_encoding,
        "unicode_normalize": normalize_unicode,
        "remove_html": remove_html,
        "remove_urls": remove_urls,
        "remove_mentions": remove_mentions,
        "remove_tatweel": remove_tatweel,
        "remove_diacritics": remove_diacritics,
        "normalize_alef": normalize_alef,
        "normalize_yeh": normalize_yeh,
        "normalize_teh_marbuta": normalize_teh_marbuta,
        "remove_extra_whitespace": remove_extra_whitespace,
    }

    rows = []
    for step_name, fn in steps.items():
        enabled = getattr(config, step_name)
        if not enabled:
            rows.append({"step": step_name, "enabled": enabled, "rows_affected": None})
            continue
        transformed = base_texts.apply(fn)
        n_affected = int((transformed != base_texts).sum())
        rows.append({"step": step_name, "enabled": enabled, "rows_affected": n_affected})

    return pd.DataFrame(rows)

summary_table = summarize_step_impact(cleaned["train"], schemas["train"]["text_col"], default_config)
print("Per-step impact on TRAIN split (single-step ablation):")
display(summary_table)


## Part 8 — Save Cleaned Datasets

Export the cleaned splits as UTF-8 CSV files, ready for tokenization / model training.


In [ ]:
OUTPUT_DIR = "clean"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATHS = {
    "train": os.path.join(OUTPUT_DIR, "train_clean.csv"),
    "dev":   os.path.join(OUTPUT_DIR, "dev_clean.csv"),
    "test":  os.path.join(OUTPUT_DIR, "test_clean.csv"),
}

for split, df in cleaned.items():
    df.to_csv(OUTPUT_PATHS[split], index=False, encoding="utf-8")
    print(f"[saved] {OUTPUT_PATHS[split]}  ({len(df)} rows, {df.shape[1]} columns)")


## Part 9 — Final Research Report

A single consolidated summary suitable for the "Dataset" section of a paper.


In [ ]:
def build_research_report(raw: dict, cleaned: dict, schemas: dict,
                           inspection_reports: dict) -> pd.DataFrame:
    rows = []
    for split in raw:
        df_raw = raw[split]
        df_clean = cleaned[split]
        text_col = schemas[split]["text_col"]
        label_cols = schemas[split]["label_cols"]

        total_rows = len(df_raw)
        dup_rows_removed = int(df_raw.duplicated().sum())
        missing_values = inspection_reports[split]["missing_values_total"]

        lengths = df_clean["text_clean"].astype(str).str.len()
        avg_len = float(lengths.mean())
        median_len = float(lengths.median())

        vocab = set()
        for t in df_clean["text_clean"].astype(str):
            vocab.update(t.split())
        vocab_size = len(vocab)

        label_dist = {}
        for col in label_cols:
            label_dist[col] = df_raw[col].value_counts().to_dict()

        rows.append({
            "split": split,
            "total_rows": total_rows,
            "duplicate_rows_found": dup_rows_removed,
            "missing_values": missing_values,
            "avg_text_length_chars": round(avg_len, 2),
            "median_text_length_chars": median_len,
            "vocabulary_size": vocab_size,
            "label_distribution": json.dumps(label_dist, ensure_ascii=False),
        })

    return pd.DataFrame(rows)


research_report = build_research_report(raw, cleaned, schemas, inspection_reports)

print("=" * 70)
print("RESEARCH REPORT — OSACT5 DATASET SUMMARY")
print("=" * 70)
display(research_report)

research_report.to_csv(os.path.join(OUTPUT_DIR, "research_report.csv"), index=False, encoding="utf-8")
print(f"\n[saved] {os.path.join(OUTPUT_DIR, 'research_report.csv')}")


## Part 10 — Load Stage-1 Outputs & Build the AMAN-LLM Layer1 Dataset

This section continues directly from the cleaned splits produced above
(`train_clean.csv`, `dev_clean.csv`, `test_clean.csv`). We (re)load them from disk so
this half of the notebook is independently re-runnable, then:

1. apply a small set of additional, scientifically-motivated refinements on top of the
   Stage-1 `text_clean` column (leading `RT`, hashtag symbol, line breaks, invisible
   Unicode characters, duplicated whitespace);
2. analyze the fine-grained `hs_label` classes;
3. collapse them into the binary `layer1_label` (`safe` / `hate_speech`) used by
   **AMAN-LLM Layer 1** (input sanitizer + intent/adversarial classifier);
4. assemble, validate, describe, and save the final Layer1 dataset together with a
   publication-ready report.

Everything below is deterministic given the Stage-1 outputs and reuses the same
`SEED`, regex, and helper-function conventions established earlier in the notebook.


In [ ]:
# Reload Stage-1 outputs (self-contained: works even in a fresh runtime)
LAYER1_INPUT_DIR = OUTPUT_DIR if "OUTPUT_DIR" in globals() else "clean"
LAYER1_INPUT_PATHS = {
    "train": os.path.join(LAYER1_INPUT_DIR, "train_clean.csv"),
    "dev":   os.path.join(LAYER1_INPUT_DIR, "dev_clean.csv"),
    "test":  os.path.join(LAYER1_INPUT_DIR, "test_clean.csv"),
}


def load_clean_split(path: str) -> pd.DataFrame:
    '''Load a previously-cleaned split, keeping every field as string (labels included).'''
    return pd.read_csv(path, encoding="utf-8", dtype=str, keep_default_na=False)


cleaned = {split: load_clean_split(path) for split, path in LAYER1_INPUT_PATHS.items()}
for split, df in cleaned.items():
    print(f"[{split}] loaded {len(df):,} rows from {LAYER1_INPUT_PATHS[split]} "
          f"| columns: {list(df.columns)}")

# Resolve schema again so this section works even if `schemas` isn't in memory
if "resolve_schema" not in globals():
    ID_CANDIDATES = ["tweet_id", "id", "twitter_id"]
    TEXT_CANDIDATES = ["text", "tweet", "tweet_text", "content"]
    LABEL_CANDIDATES = ["off_label", "hs_label", "offensive_label", "label", "off", "hs",
                         "hate_speech_label", "hate_speech", "class"]

    def find_column(columns, candidates):
        lower_map = {c.lower(): c for c in columns}
        for cand in candidates:
            if cand in lower_map:
                return lower_map[cand]
        return None

    def resolve_schema(df: pd.DataFrame) -> dict:
        id_col = find_column(df.columns, ID_CANDIDATES)
        text_col = find_column(df.columns, TEXT_CANDIDATES)
        label_cols = [c for c in df.columns if c.lower() in LABEL_CANDIDATES]
        if text_col is None:
            text_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]
        if id_col is None and len(df.columns) > 1:
            id_col = df.columns[0]
        return {"id_col": id_col, "text_col": text_col, "label_cols": label_cols}

schemas = {split: resolve_schema(df) for split, df in cleaned.items()}
for split, schema in schemas.items():
    print(f"{split:6s} -> {schema}")


## Part 11 — Layer1-Specific Preprocessing Refinements

The Stage-1 pipeline (Part 6) already handles encoding repair, Unicode normalization,
URL/mention/HTML removal, tatweel/diacritics, and Alef/Yeh normalization. For the
Layer1 dataset we add a small number of **additional, narrowly-scoped** refinements on
top of the existing `text_clean` column:

| Step | Rationale |
|---|---|
| Remove leading `RT` | Purely a retweet marker, carries no semantic/intent signal. |
| Strip `#` (keep hashtag text) | The hashtag *content* (e.g. `بايع_الكليجا`) is meaningful text; the `#` glyph itself is not. |
| Collapse line breaks to a single space | Prevents a tweet from being spuriously split into multiple "sentences" by downstream tokenizers. |
| Remove invisible Unicode characters | Zero-width spaces/joiners, bidi-control marks, BOM, soft hyphen — these are rendering artifacts, not content. |
| Re-apply NFKC Unicode normalization | Idempotent safety net; guarantees Layer1 output is canonical even if this section is run standalone. |
| Collapse duplicated whitespace | Cosmetic only, applied last. |

We deliberately **do not** touch Arabic words, Darja, Arabizi, French words, emojis,
numbers, or punctuation — all of these carry information relevant to intent/adversarial
detection in AMAN-LLM Layer 1.


In [ ]:
# Layer1-specific refinement functions (composable, single-purpose)
RT_LEADING_RE = re.compile(r"^\s*rt\s*[:\-]?\s*", re.IGNORECASE)
HASHTAG_SYMBOL_RE = re.compile(r"#(?=\S)")
LINEBREAK_RE = re.compile(r"[\r\n]+")
# zero-width space/joiner/non-joiner, bidi-control marks, word joiner, invisible math
# operators, soft hyphen, byte-order mark
INVISIBLE_CHARS_RE = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2060-\u2064\uFEFF\u00AD]")
if "WHITESPACE_RE" not in globals():
    WHITESPACE_RE = re.compile(r"\s+")
if "normalize_unicode" not in globals():
    def normalize_unicode(text: str) -> str:
        return unicodedata.normalize("NFKC", text)


def remove_leading_rt(text: str) -> str:
    '''Remove a single leading "RT" retweet marker (optionally followed by ':' or '-').'''
    return RT_LEADING_RE.sub("", text, count=1)


def strip_hashtag_symbol(text: str) -> str:
    '''Drop the '#' glyph but keep the hashtag text itself, e.g. #بايع_الكليجا -> بايع_الكليجا.'''
    return HASHTAG_SYMBOL_RE.sub("", text)


def normalize_linebreaks(text: str) -> str:
    '''Collapse any CR/LF sequence into a single space so a tweet stays one line.'''
    return LINEBREAK_RE.sub(" ", text)


def remove_invisible_chars(text: str) -> str:
    '''Strip zero-width / bidi-control / BOM characters that carry no semantic content.'''
    return INVISIBLE_CHARS_RE.sub("", text)


def dedupe_spaces(text: str) -> str:
    '''Collapse runs of whitespace into a single space and trim the ends.'''
    return WHITESPACE_RE.sub(" ", text).strip()


# Fixed, documented order: structural markers first, then hashtag symbol, then a final
# Unicode re-normalization + whitespace collapse.
LAYER1_STEPS = [
    ("remove_leading_rt", remove_leading_rt),
    ("normalize_linebreaks", normalize_linebreaks),
    ("remove_invisible_chars", remove_invisible_chars),
    ("strip_hashtag_symbol", strip_hashtag_symbol),
    ("unicode_normalize", normalize_unicode),
    ("dedupe_spaces", dedupe_spaces),
]


def layer1_clean_text(text: str) -> str:
    '''Apply the Layer1 refinement steps on top of the Stage-1 `text_clean` value.

    Preserves Arabic, Darja, Arabizi, French words, emojis, numbers, and punctuation.
    '''
    if not isinstance(text, str):
        return text
    for _, fn in LAYER1_STEPS:
        text = fn(text)
    return text


# Apply to every split, keeping the pre-refinement text so we can report deltas
layer1_before = {}
for split, df in cleaned.items():
    base_col = "text_clean" if "text_clean" in df.columns else schemas[split]["text_col"]
    layer1_before[split] = df[base_col].astype(str).copy()
    df["text_clean"] = layer1_before[split].apply(layer1_clean_text)

for split, df in cleaned.items():
    n_changed = int((df["text_clean"] != layer1_before[split]).sum())
    print(f"[{split}] Layer1 refinement changed {n_changed:,} / {len(df):,} rows "
          f"({n_changed / len(df):.1%})")


In [ ]:
# Qualitative before/after examples for the Layer1 refinement step
def show_layer1_before_after(df: pd.DataFrame, before: pd.Series, n: int = 6, seed: int = SEED) -> None:
    changed_idx = df.index[df["text_clean"] != before]
    if len(changed_idx) == 0:
        print("No rows were changed by the Layer1 refinement step.")
        return
    sample_idx = pd.Series(changed_idx).sample(min(n, len(changed_idx)), random_state=seed)
    for i in sample_idx:
        print("-" * 60)
        print("BEFORE:", before.loc[i])
        print("AFTER :", df.loc[i, "text_clean"])


print("Layer1 refinement examples (train split):")
show_layer1_before_after(cleaned["train"], layer1_before["train"])


## Part 12 — Hate Speech Label Analysis

We inspect the fine-grained OSACT hate-speech column, print per-class counts for the
target classes (`NOT_HS`, `HS1`, `HS2`, `HS3`, `HS5`, `HS6`) on every labeled split, flag
any unexpected label values (e.g. `HS4`, typos, or annotation artifacts) separately, and
visualize the distribution. The (unlabeled) test split is skipped automatically.


In [ ]:
# Resolve the hate-speech label column robustly across possible naming conventions
HS_LABEL_CANDIDATES = ["hs_label", "hate_speech", "hate_speech_label", "hs", "hatespeech"]
EXPECTED_HS_CLASSES = ["NOT_HS", "HS1", "HS2", "HS3", "HS5", "HS6"]


def resolve_hs_column(df: pd.DataFrame, schema: dict):
    '''Find the hate-speech label column by name first, then by value overlap as a fallback.'''
    col = find_column(df.columns, HS_LABEL_CANDIDATES)
    if col:
        return col
    for candidate in schema["label_cols"]:
        values = set(df[candidate].astype(str).str.upper().unique())
        if values & set(EXPECTED_HS_CLASSES):
            return candidate
    return None


hs_columns = {split: resolve_hs_column(df, schemas[split]) for split, df in cleaned.items()}
for split, col in hs_columns.items():
    print(f"{split:6s} -> hate-speech column: {col!r}")


def print_hs_counts(df: pd.DataFrame, col: str, split_name: str) -> pd.Series:
    print("=" * 60)
    print(f"{split_name}: hate-speech label counts (column '{col}')")
    print("=" * 60)
    counts = df[col].astype(str).str.upper().value_counts()
    for cls in EXPECTED_HS_CLASSES:
        print(f"  {cls:8s}: {int(counts.get(cls, 0)):6,d}")
    unexpected = counts[~counts.index.isin(EXPECTED_HS_CLASSES)]
    if len(unexpected):
        print("  Unexpected/extra label values found:")
        print("  " + unexpected.to_string().replace("\n", "\n  "))
    return counts


hs_counts = {}
for split, df in cleaned.items():
    col = hs_columns[split]
    if col is None:
        print(f"[{split}] no hate-speech column detected — skipping "
              f"(expected for an unlabeled test split).")
        continue
    hs_counts[split] = print_hs_counts(df, col, split)


In [ ]:
# Bar charts of the hate-speech class distribution
def plot_hs_distribution(counts: pd.Series, split_name: str) -> None:
    ordered = counts.reindex(EXPECTED_HS_CLASSES, fill_value=0)
    plt.figure(figsize=(7, 4))
    bars = plt.bar(ordered.index, ordered.values, color="#3B6E8F")
    plt.title(f"{split_name}: hate-speech label distribution")
    plt.xlabel("Class")
    plt.ylabel("Count")
    for bar, v in zip(bars, ordered.values):
        plt.text(bar.get_x() + bar.get_width() / 2, v, f"{v:,}",
                  ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.show()


for split, counts in hs_counts.items():
    plot_hs_distribution(counts, split)


## Part 13 — Create the `layer1_label` Column

We map every fine-grained hate-speech class onto the binary Layer1 target:

```
NOT_HS                          -> safe
HS1, HS2, HS3, HS5, HS6         -> hate_speech
```

The original `hs_label` column (or equivalent) is **not** overwritten — both columns
coexist in `cleaned[split]`. Any value outside the expected set maps to `NaN` so it
surfaces during validation (Part 15) instead of being silently mislabeled.


In [ ]:
# NOT_HS -> safe, every HS* sub-class -> hate_speech (original column kept intact)
LAYER1_LABEL_MAP = {"NOT_HS": "safe"}
for hs_class in EXPECTED_HS_CLASSES:
    if hs_class != "NOT_HS":
        LAYER1_LABEL_MAP[hs_class] = "hate_speech"


def map_to_layer1_label(value, mapping=LAYER1_LABEL_MAP):
    '''Map a raw OSACT hate-speech label to the coarse Layer1 label.

    Unmapped/unexpected values become NaN rather than being silently guessed at, so they
    are caught by the Part 15 validation step.
    '''
    return mapping.get(str(value).upper(), np.nan)


for split, df in cleaned.items():
    col = hs_columns.get(split)
    if col is None:
        df["layer1_label"] = np.nan  # unlabeled split (test)
        print(f"[{split}] no gold labels available — layer1_label set to NaN for all rows.")
        continue
    df["layer1_label"] = df[col].apply(map_to_layer1_label)
    print(f"[{split}] layer1_label distribution:")
    print(df["layer1_label"].value_counts(dropna=False).to_string())
    print()


## Part 14 — Build the Layer1 Dataset

In [ ]:
# Project each split down to the Layer1 schema
def build_layer1_dataset(df: pd.DataFrame, schema: dict) -> pd.DataFrame:
    '''Return a new dataframe with exactly: tweet_id, text, text_clean, layer1_label, source.'''
    id_col = schema["id_col"]
    out = pd.DataFrame({
        "tweet_id": df[id_col].values if id_col else np.arange(len(df)),
        "text": df[schema["text_col"]].values,
        "text_clean": df["text_clean"].values,
        "layer1_label": df["layer1_label"].values,
    })
    out["source"] = "OSACT"
    return out


layer1_datasets = {split: build_layer1_dataset(df, schemas[split]) for split, df in cleaned.items()}
for split, df in layer1_datasets.items():
    print(f"[{split}] Layer1 dataset shape: {df.shape}")
    display(df.head())


## Part 15 — Layer1 Dataset Validation

We verify, per split: no missing values, no duplicated `tweet_id`, no duplicated
`text_clean`, and (for labeled splits) that `layer1_label` only takes the values
`safe` / `hate_speech`. The unlabeled test split is validated on everything except
label validity, which is expected to be all-`NaN` by design.


In [ ]:
# Layer1 dataset validation
VALID_LAYER1_LABELS = {"safe", "hate_speech"}


def validate_layer1_dataset(df: pd.DataFrame, split_name: str, labels_expected: bool) -> dict:
    issues, notes = [], []

    n_missing_by_col = df.isna().sum()
    n_missing_by_col = n_missing_by_col[n_missing_by_col > 0]
    # For an unlabeled split, an all-NaN layer1_label is expected, not an error.
    if not labels_expected:
        n_missing_by_col = n_missing_by_col.drop(labels=["layer1_label"], errors="ignore")
        notes.append("unlabeled split (test) — layer1_label is NaN by design and excluded "
                     "from the missing-value / label-validity checks")
    n_missing = int(n_missing_by_col.sum())
    if n_missing:
        issues.append(f"{n_missing} missing value(s): {n_missing_by_col.to_dict()}")

    dup_ids = int(df["tweet_id"].duplicated().sum())
    if dup_ids:
        issues.append(f"{dup_ids} duplicated tweet_id value(s)")

    dup_texts = int(df["text_clean"].duplicated().sum())
    if dup_texts:
        issues.append(f"{dup_texts} duplicated text_clean value(s)")

    if labels_expected:
        invalid_labels = df.loc[~df["layer1_label"].isin(VALID_LAYER1_LABELS)]
        if len(invalid_labels):
            issues.append(f"{len(invalid_labels)} row(s) with an invalid/missing "
                           f"layer1_label (expected one of {sorted(VALID_LAYER1_LABELS)})")

    return {
        "split": split_name,
        "n_rows": len(df),
        "n_missing_values": n_missing,
        "n_duplicate_ids": dup_ids,
        "n_duplicate_texts": dup_texts,
        "issues": issues,
        "notes": notes,
        "passed": len(issues) == 0,
    }


validation_reports = {}
for split, df in layer1_datasets.items():
    labels_expected = hs_columns.get(split) is not None
    report = validate_layer1_dataset(df, split, labels_expected)
    validation_reports[split] = report
    print("=" * 60)
    print(f"VALIDATION REPORT — {split}  (PASSED: {report['passed']})")
    print("=" * 60)
    for key in ["n_rows", "n_missing_values", "n_duplicate_ids", "n_duplicate_texts"]:
        print(f"{key:18s}: {report[key]}")
    print(f"{'issues':18s}: {report['issues'] or 'none'}")
    print(f"{'notes':18s}: {report['notes'] or 'none'}")
    print()


## Part 16 — Layer1 Dataset Statistics

In [ ]:
# Layer1 dataset statistics
DIGIT_RE = re.compile(r"[0-9\u0660-\u0669]")  # ASCII digits + Arabic-Indic digits
if "ARABIC_CHAR_RE" not in globals():
    ARABIC_CHAR_RE = re.compile(r"[\u0600-\u06FF\u0750-\u077F]")
if "LATIN_CHAR_RE" not in globals():
    LATIN_CHAR_RE = re.compile(r"[A-Za-z]")

if "count_emojis" not in globals():
    def count_emojis(text: str) -> int:
        return len([ch for ch in text if ch in emoji.EMOJI_DATA])


def compute_layer1_statistics(df: pd.DataFrame, split_name: str) -> dict:
    texts = df["text_clean"].astype(str)
    char_lens = texts.str.len()
    token_lens = texts.str.split().apply(len)

    vocab = set()
    for t in texts:
        vocab.update(t.split())

    def char_ratio(pattern: re.Pattern) -> float:
        return float(texts.apply(lambda t: len(pattern.findall(t)) / max(len(t), 1)).mean())

    emoji_counts = texts.apply(count_emojis)

    return {
        "split": split_name,
        "total_samples": len(df),
        "safe_samples": int((df["layer1_label"] == "safe").sum()),
        "hate_speech_samples": int((df["layer1_label"] == "hate_speech").sum()),
        "avg_char_length": round(float(char_lens.mean()), 2),
        "avg_token_length": round(float(token_lens.mean()), 2),
        "median_token_length": float(token_lens.median()),
        "vocabulary_size": len(vocab),
        "arabic_char_ratio": round(char_ratio(ARABIC_CHAR_RE), 4),
        "latin_char_ratio": round(char_ratio(LATIN_CHAR_RE), 4),
        "digit_ratio": round(char_ratio(DIGIT_RE), 4),
        "avg_emoji_per_sample": round(float(emoji_counts.mean()), 4),
        "total_emojis": int(emoji_counts.sum()),
    }


layer1_stats = {split: compute_layer1_statistics(df, split) for split, df in layer1_datasets.items()}
layer1_stats_df = pd.DataFrame(layer1_stats.values())

print("=" * 70)
print("LAYER1 DATASET STATISTICS")
print("=" * 70)
display(layer1_stats_df)


## Part 17 — Save the Layer1 Datasets

In [ ]:
LAYER1_OUTPUT_DIR = "layer1"
os.makedirs(LAYER1_OUTPUT_DIR, exist_ok=True)

LAYER1_OUTPUT_PATHS = {
    "train": os.path.join(LAYER1_OUTPUT_DIR, "train_layer1.csv"),
    "dev":   os.path.join(LAYER1_OUTPUT_DIR, "dev_layer1.csv"),
    "test":  os.path.join(LAYER1_OUTPUT_DIR, "test_layer1.csv"),
}

for split, df in layer1_datasets.items():
    df.to_csv(LAYER1_OUTPUT_PATHS[split], index=False, encoding="utf-8")
    print(f"[saved] {LAYER1_OUTPUT_PATHS[split]}  ({len(df)} rows, {df.shape[1]} columns)")


## Part 18 — Layer1 Research Report

A single, flat (`section, split, metric, value`) report combining the dataset summary,
label distribution, cleaning statistics, validation results, and a few concrete
preprocessing examples — saved as `layer1_report.csv` for direct inclusion in the
AMAN-LLM thesis/paper appendix.


In [ ]:
def build_layer1_report(layer1_stats: dict, validation_reports: dict,
                         layer1_before: dict, layer1_datasets: dict,
                         cleaned: dict, n_examples: int = 3) -> pd.DataFrame:
    rows = []

    # -- Dataset summary --
    for split, stats in layer1_stats.items():
        for metric, value in stats.items():
            if metric == "split":
                continue
            rows.append({"section": "dataset_summary", "split": split,
                         "metric": metric, "value": value})

    # -- Label distribution --
    for split, df in layer1_datasets.items():
        for label, count in df["layer1_label"].value_counts(dropna=False).items():
            rows.append({"section": "label_distribution", "split": split,
                         "metric": str(label), "value": int(count)})

    # -- Cleaning statistics (Layer1 refinement step) --
    for split, before in layer1_before.items():
        after = cleaned[split]["text_clean"]
        rows.append({"section": "cleaning_statistics", "split": split,
                     "metric": "rows_changed_by_layer1_refinement",
                     "value": int((after != before).sum())})
        rows.append({"section": "cleaning_statistics", "split": split,
                     "metric": "total_rows", "value": len(before)})

    # -- Validation results --
    for split, report in validation_reports.items():
        for metric in ["passed", "n_missing_values", "n_duplicate_ids", "n_duplicate_texts"]:
            rows.append({"section": "validation", "split": split,
                         "metric": metric, "value": report[metric]})
        rows.append({"section": "validation", "split": split, "metric": "issues",
                     "value": "; ".join(report["issues"]) if report["issues"] else "none"})

    # -- Preprocessing examples (train split) --
    before_train = layer1_before["train"]
    after_train = cleaned["train"]["text_clean"]
    changed_idx = before_train.index[after_train != before_train][:n_examples]
    for rank, i in enumerate(changed_idx, start=1):
        rows.append({"section": "preprocessing_examples", "split": "train",
                     "metric": f"example_{rank}_before", "value": before_train.loc[i]})
        rows.append({"section": "preprocessing_examples", "split": "train",
                     "metric": f"example_{rank}_after", "value": after_train.loc[i]})

    return pd.DataFrame(rows, columns=["section", "split", "metric", "value"])


layer1_report = build_layer1_report(layer1_stats, validation_reports, layer1_before,
                                     layer1_datasets, cleaned)

print("=" * 70)
print("LAYER1 RESEARCH REPORT")
print("=" * 70)
display(layer1_report)

layer1_report_path = os.path.join(LAYER1_OUTPUT_DIR, "layer1_report.csv")
layer1_report.to_csv(layer1_report_path, index=False, encoding="utf-8")
print(f"\n[saved] {layer1_report_path}")


## Part 19 — Final Scientific Validation of the OSACT Layer1 Dataset (Pre-Merge QA)

### Part 19.1 — Verify the `layer1_label` Mapping

We re-derive `layer1_label` independently from the raw hate-speech column and compare it
against what Parts 10–18 produced, printing a diagnostic of the raw label values first.
If the two disagree (e.g. because of stray whitespace, casing, or a mis-resolved column),
we report exactly what was wrong and re-apply a corrected, whitespace/case-robust mapping
before proceeding — this is a bug fix to the label *derivation*, not a change to the
underlying data.


In [ ]:
assert "cleaned" in globals() and "layer1_datasets" in globals() and "hs_columns" in globals(), (
    "Part 19 assumes Parts 10-18 already ran in this session (cleaned / layer1_datasets / "
    "hs_columns / schemas in memory). Run those cells first if this is a fresh runtime."
)

# Defensive re-resolution: scan every column, not just the previously-detected ones
def resolve_hs_column_strict(df: pd.DataFrame):
    '''Find the hate-speech column by value overlap across ALL columns, whitespace/case-insensitive.'''
    for c in df.columns:
        values = set(df[c].astype(str).str.strip().str.upper().unique())
        if values & set(EXPECTED_HS_CLASSES):
            return c
    return None

for split, df in cleaned.items():
    if hs_columns.get(split) is None:
        fallback = resolve_hs_column_strict(df)
        if fallback:
            print(f"[{split}] hate-speech column re-resolved via full-column scan: "
                  f"'{fallback}' (previous resolution returned None)")
            hs_columns[split] = fallback

# Raw value diagnostic: exactly what is actually in the label column
for split, col in hs_columns.items():
    df = cleaned[split]
    if col is None:
        print(f"[{split}] no hate-speech column resolved — skipping raw-label diagnostic.")
        continue
    print("=" * 60)
    print(f"{split}: RAW values in hate-speech column '{col}' (top 20)")
    print("=" * 60)
    raw_counts = df[col].value_counts(dropna=False)
    print(raw_counts.head(20).to_string())
    print(f"  -> {int(raw_counts.sum())} total rows, {int(df[col].isna().sum())} NaN")
    print()


In [ ]:
# Re-derive layer1_label with a whitespace/case-robust mapping and reconcile
def robust_map_to_layer1_label(value, mapping=LAYER1_LABEL_MAP):
    if pd.isna(value):
        return np.nan
    return mapping.get(str(value).strip().upper(), np.nan)


label_fix_report = []
for split, df in cleaned.items():
    col = hs_columns.get(split)
    if col is None:
        continue

    recomputed = df[col].apply(robust_map_to_layer1_label)
    existing = df["layer1_label"] if "layer1_label" in df.columns else pd.Series(
        [np.nan] * len(df), index=df.index)

    n_mismatch = int((recomputed.fillna("<NA>") != existing.fillna("<NA>")).sum())
    label_fix_report.append({
        "split": split,
        "n_mismatch": n_mismatch,
        "existing_non_null": int(existing.notna().sum()),
        "recomputed_non_null": int(recomputed.notna().sum()),
    })

    if n_mismatch:
        print(f"[{split}] found {n_mismatch} row(s) where the stored layer1_label "
              f"disagreed with a fresh whitespace/case-robust re-derivation from '{col}'. "
              f"Correcting layer1_label for this split (source label column is untouched).")
        df["layer1_label"] = recomputed
        if split in layer1_datasets:
            layer1_datasets[split]["layer1_label"] = recomputed.values
    else:
        print(f"[{split}] OK — stored layer1_label matches a fresh re-derivation "
              f"({int(existing.notna().sum())} labeled rows).")

print()
display(pd.DataFrame(label_fix_report))


### Part 19.2 — Label Counts, Percentages, and Distribution Plot

With `layer1_label` verified (and corrected above if needed), we print counts and
percentages per split and confirm the only values present are `safe` / `hate_speech`
(and `NaN` for the unlabeled test split).


In [ ]:
def layer1_label_report(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    counts = df["layer1_label"].value_counts(dropna=False)
    pct = (counts / len(df) * 100).round(2)
    report = pd.DataFrame({"count": counts, "percent": pct})

    print("=" * 60)
    print(f"{split_name}: layer1_label counts & percentages")
    print("=" * 60)
    display(report)

    unexpected = [v for v in counts.index if pd.notna(v) and v not in ("safe", "hate_speech")]
    if unexpected:
        print(f"  \u26a0 unexpected label value(s) found: {unexpected}")
    else:
        print("  OK \u2014 no unexpected label values (only 'safe' / 'hate_speech', "
              "plus NaN for the unlabeled test split).")
    print()
    return report


layer1_label_reports = {split: layer1_label_report(df, split) for split, df in layer1_datasets.items()}


def plot_layer1_label_distribution(df: pd.DataFrame, split_name: str) -> None:
    counts = df["layer1_label"].value_counts(dropna=False)
    labels = [str(l) for l in counts.index]
    palette = {"safe": "#3B6E8F", "hate_speech": "#8F3B5C", "nan": "#999999"}
    colors = [palette.get(l, "#999999") for l in labels]

    plt.figure(figsize=(5, 4))
    bars = plt.bar(labels, counts.values, color=colors)
    plt.title(f"{split_name}: layer1_label distribution")
    plt.xlabel("Label")
    plt.ylabel("Count")
    for bar, v in zip(bars, counts.values):
        plt.text(bar.get_x() + bar.get_width() / 2, v, f"{v:,}",
                  ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.show()


for split, df in layer1_datasets.items():
    plot_layer1_label_distribution(df, split)


## Part 20 — Dataset Quality Metrics

Independent quality metrics on the final Layer1 dataset: uniqueness of `text_clean`,
duplication (cleaned vs. original text), token-length statistics, and vocabulary size.


In [ ]:
def dataset_quality_report(df: pd.DataFrame, split_name: str) -> dict:
    token_lens = df["text_clean"].astype(str).str.split().apply(len)

    vocab = set()
    for t in df["text_clean"].astype(str):
        vocab.update(t.split())

    return {
        "split": split_name,
        "n_rows": len(df),
        "n_unique_texts": int(df["text_clean"].nunique()),
        "n_duplicated_cleaned_texts": int(df["text_clean"].duplicated().sum()),
        "n_duplicated_original_texts": int(df["text"].duplicated().sum()),
        "avg_token_length": round(float(token_lens.mean()), 2),
        "median_token_length": float(token_lens.median()),
        "max_token_length": int(token_lens.max()),
        "vocabulary_size": len(vocab),
    }


quality_reports = {split: dataset_quality_report(df, split) for split, df in layer1_datasets.items()}
quality_df = pd.DataFrame(quality_reports.values())

print("=" * 70)
print("DATASET QUALITY REPORT")
print("=" * 70)
display(quality_df)


## Part 21 — Language / Script Analysis

Per-tweet script flags (Arabic script present, Latin script present, both present,
digits present, at least one emoji, code-switched at the *token* level — i.e. at least
one Arabic-script token and one Latin-script token in the same tweet), aggregated into
per-split percentages.


In [ ]:
# Fallbacks in case this section is run standalone
if "ARABIC_CHAR_RE" not in globals():
    ARABIC_CHAR_RE = re.compile(r"[\u0600-\u06FF\u0750-\u077F]")
if "LATIN_CHAR_RE" not in globals():
    LATIN_CHAR_RE = re.compile(r"[A-Za-z]")
if "DIGIT_RE" not in globals():
    DIGIT_RE = re.compile(r"[0-9\u0660-\u0669]")
if "count_emojis" not in globals():
    def count_emojis(text: str) -> int:
        return len([ch for ch in text if ch in emoji.EMOJI_DATA])


def script_flags(text: str) -> pd.Series:
    has_arabic = bool(ARABIC_CHAR_RE.search(text))
    has_latin = bool(LATIN_CHAR_RE.search(text))
    has_digit = bool(DIGIT_RE.search(text))
    has_emoji = count_emojis(text) > 0

    tokens = text.split()
    arabic_tokens = sum(1 for tok in tokens if ARABIC_CHAR_RE.search(tok))
    latin_tokens = sum(1 for tok in tokens if LATIN_CHAR_RE.search(tok))
    code_switched = arabic_tokens > 0 and latin_tokens > 0

    return pd.Series({
        "has_arabic": has_arabic,
        "has_latin": has_latin,
        "mixed_script": has_arabic and has_latin,
        "has_digit": has_digit,
        "has_emoji": has_emoji,
        "code_switched": code_switched,
    })


def language_analysis(df: pd.DataFrame, split_name: str):
    flags = df["text_clean"].astype(str).apply(script_flags)
    summary = {
        "split": split_name,
        "pct_arabic_script": round(100 * flags["has_arabic"].mean(), 2),
        "pct_latin_script": round(100 * flags["has_latin"].mean(), 2),
        "pct_mixed_arabic_latin": round(100 * flags["mixed_script"].mean(), 2),
        "pct_contains_digits": round(100 * flags["has_digit"].mean(), 2),
        "pct_emoji_containing": round(100 * flags["has_emoji"].mean(), 2),
        "pct_code_switched": round(100 * flags["code_switched"].mean(), 2),
    }
    return summary, flags


language_summaries, language_flags = {}, {}
for split, df in layer1_datasets.items():
    summary, flags = language_analysis(df, split)
    language_summaries[split] = summary
    language_flags[split] = flags

language_summary_df = pd.DataFrame(language_summaries.values())
print("=" * 70)
print("LANGUAGE / SCRIPT ANALYSIS (% of tweets)")
print("=" * 70)
display(language_summary_df)


## Part 22 — Text Complexity

Average words/sentences per tweet, token- and character-length distributions, and
Zipf vocabulary statistics (rank-frequency slope via a log-log least-squares fit).


In [ ]:
SENTENCE_SPLIT_RE = re.compile(r"[.!\u061F?]+")


def complexity_metrics(df: pd.DataFrame, split_name: str):
    texts = df["text_clean"].astype(str)
    words = texts.str.split().apply(len)
    sentences = texts.apply(
        lambda t: max(1, len([s for s in SENTENCE_SPLIT_RE.split(t) if s.strip()]))
    )
    words_per_sentence = words / sentences
    chars = texts.str.len()

    summary = {
        "split": split_name,
        "avg_words_per_tweet": round(float(words.mean()), 2),
        "avg_sentences_per_tweet": round(float(sentences.mean()), 2),
        "avg_words_per_sentence": round(float(words_per_sentence.mean()), 2),
        "avg_chars_per_tweet": round(float(chars.mean()), 2),
    }
    return summary, words, chars


complexity_summaries, token_dists, char_dists = {}, {}, {}
for split, df in layer1_datasets.items():
    summary, words, chars = complexity_metrics(df, split)
    complexity_summaries[split] = summary
    token_dists[split] = words
    char_dists[split] = chars

complexity_df = pd.DataFrame(complexity_summaries.values())
print("=" * 70)
print("TEXT COMPLEXITY SUMMARY")
print("=" * 70)
display(complexity_df)


In [ ]:
# Token- and character-length distributions (train split)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(token_dists["train"], bins=30, color="#3B6E8F", edgecolor="white")
axes[0].set_title("train: token-length distribution")
axes[0].set_xlabel("Tokens per tweet")
axes[0].set_ylabel("Frequency")

axes[1].hist(char_dists["train"], bins=30, color="#8F3B5C", edgecolor="white")
axes[1].set_title("train: character-length distribution")
axes[1].set_xlabel("Characters per tweet")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


In [ ]:
# Zipf vocabulary statistics
def zipf_statistics(df: pd.DataFrame, split_name: str):
    tokens = []
    for t in df["text_clean"].astype(str):
        tokens.extend(t.split())

    freq = pd.Series(Counter(tokens)).sort_values(ascending=False)
    ranks = np.arange(1, len(freq) + 1)
    # Least-squares fit of log(frequency) ~ slope * log(rank) + intercept.
    # An ideal Zipfian distribution has slope ~= -1.
    slope, intercept = np.polyfit(np.log(ranks), np.log(freq.values), 1)

    summary = {
        "split": split_name,
        "vocabulary_size": len(freq),
        "total_tokens": int(freq.sum()),
        "zipf_slope": round(float(slope), 3),
        "top_5_tokens": freq.head(5).to_dict(),
    }
    return summary, freq


zipf_summaries, zipf_freqs = {}, {}
for split, df in layer1_datasets.items():
    summary, freq = zipf_statistics(df, split)
    zipf_summaries[split] = summary
    zipf_freqs[split] = freq

print("=" * 70)
print("ZIPF VOCABULARY STATISTICS")
print("=" * 70)
display(pd.DataFrame(zipf_summaries.values()))

plt.figure(figsize=(6, 5))
freq = zipf_freqs["train"]
ranks = np.arange(1, len(freq) + 1)
plt.loglog(ranks, freq.values, marker=".", linestyle="none", color="#3B6E8F")
plt.title("train: Zipf plot (log rank vs. log frequency)")
plt.xlabel("Rank (log scale)")
plt.ylabel("Frequency (log scale)")
plt.tight_layout()
plt.show()


## Part 23 — Class Balance Analysis & Training Recommendations

We quantify the `safe` / `hate_speech` imbalance and compute inverse-frequency class
weights for reference. **No resampling or reweighting is applied to the data here** —
this section only reports numbers and a recommendation; the exported files in Part 24
retain the original, unmodified label distribution.


In [ ]:
def class_balance_report(df: pd.DataFrame, split_name: str):
    counts = df["layer1_label"].value_counts(dropna=True)
    if counts.empty:
        return None

    total = int(counts.sum())
    ratio = float(counts.max() / counts.min()) if counts.min() > 0 else float("inf")
    # Standard inverse-frequency weighting, normalized so the mean weight is ~1.
    weights = (total / (len(counts) * counts)).round(3).to_dict()

    return {
        "split": split_name,
        "n_labeled": total,
        "class_counts": counts.to_dict(),
        "imbalance_ratio_majority_to_minority": round(ratio, 2),
        "suggested_inverse_freq_class_weights": weights,
    }


balance_reports = {}
for split, df in layer1_datasets.items():
    report = class_balance_report(df, split)
    if report is None:
        continue
    balance_reports[split] = report
    print("=" * 60)
    print(f"{split}: class balance")
    print("=" * 60)
    for key, value in report.items():
        print(f"{key:40s}: {value}")
    print()


## Part 24 — Export the Official OSACT Layer1 Datasets

With label correctness, quality, language composition, complexity, and class balance
all verified, we export the validated splits as the **official** OSACT Layer1 datasets.

In [ ]:
LAYER1_FINAL_DIR = "layer1"
os.makedirs(LAYER1_FINAL_DIR, exist_ok=True)

LAYER1_FINAL_PATHS = {
    "train": os.path.join(LAYER1_FINAL_DIR, "train_layer1_final.csv"),
    "dev":   os.path.join(LAYER1_FINAL_DIR, "dev_layer1_final.csv"),
    "test":  os.path.join(LAYER1_FINAL_DIR, "test_layer1_final.csv"),
}

for split, df in layer1_datasets.items():
    df.to_csv(LAYER1_FINAL_PATHS[split], index=False, encoding="utf-8")
    print(f"[saved] {LAYER1_FINAL_PATHS[split]}  ({len(df)} rows, {df.shape[1]} columns)")

print("\nThese are the official, validated OSACT Layer1 datasets — ready for a future, "
      "separate merge step with AAAC. No merging has been performed in this notebook.")
